# Newton's Law of Cooling Simulation - Tanh RNN Case

This notebook is intended to demonstrate manually setting the weights of an RNN with **tanh activation** to reproduce a solution to Newton's Law of Cooling, then how to time warp to change the characteristic time scale. 

Continuous Solution:

$$
T(t) = e^{-kt}\;T_0 + (1 - e^{-kt})\;T_a
$$

Discretization Time Step with $\Delta t=1$, exact solution:

$$
T_{t+1} = e^{-k}\;T_{t} + (1 - e^{-k})\;T_a
$$

We will set up the recurrent layer with 1 unit to approximate the system closely. The temperature curves must be squashed into the near-linear portion of the tanh function.

Methodology Steps:
1. Pick $\mu$ and $\sigma$ for a normal distribution
2. Sample $\{T_0, T_{a}\}$ values pairs, N draws
3. Pool samples, clip to 1th and 99th percentiles (remove outliers)
4. Calculate global $T_{min}$ and $T_{max}$
5. Using global min and max, map all values to range $[-\delta, \delta]$, for a tight interval around 0
6. Evolve curves by Newton's Law in compressed space (NOTE: dynamics invariant to scale)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import tensorflow as tf
import src.reproducibility as reproducibility

def newton_sol(T, Ta, k, t):
    """
    Discretization of exact solution with T(0)=T0
    """
    return np.exp(-k*t)*T + (1-np.exp(-k*t))*Ta

## Simulate

Draw initial temp and ambient temps from shared distribution. Thus, the mean difference between the variables is zero, and the resulting curves are a mix of cooling vs heating

In [ ]:
# Sample temps for constructing curves
mu = 15  # chosen as 15C, close to global average land temp
s = 10   # std of temp
N = 1000 # number of samples

reproducibility.set_seed(123)
# sample ambient and initial temperatures independently
Ta = np.random.normal(loc=mu, scale=s, size=N)
T0 = np.random.normal(loc=mu, scale=s, size=N)

In [ ]:
# # Filter percentiles
# p05, p995 = np.percentile(np.r_[Ta, T0], q = (.5, 99.5)) # Inner 99% 
# inds = np.where((Ta>=p05) & (Ta<=p995) & (T0>=p05) & (T0<=p995))[0] 
# Ta = Ta[inds]
# T0 = T0[inds]

In [ ]:
# Map to tight interval [-delta, delta]
# Normal min/max scaling maps to [0, 1], so we need 2*delta * minmax - delta
delta = 0.5
tmin, tmax = np.min(np.r_[Ta, T0]), np.max(np.r_[Ta, T0])
x = 2*delta * (Ta - tmin) / (tmax-tmin) - delta
x0 = 2*delta * (T0 - tmin) / (tmax-tmin) - delta

In [ ]:
Nt = 25 # number of time steps to evolve temp
t = np.arange(1, Nt + 1)   # time zero is x0, evolve forward Nt steps
k1 = 0.2

X = np.repeat(x[:, None, None], Nt, axis=1).astype(np.float32)
expkt = np.exp(-k1 * t)[None, :]     # (1, Nt)
y = expkt * x0[:, None] + (1 - expkt) * x[:, None] # shape: (N, Nt)
y = y[..., None]
y = np.concatenate([x0[:, None, None] , y], axis=1)

In [ ]:
# Plot a few curves
fig, ax = plt.subplots(nrows = 3, ncols= 3, sharex=True, sharey=True)
for i in range(0, 9):
    ax.flat[i].plot(y[i, ...])
fig.suptitle("Sample Temp Curves")

In [ ]:
# Visualize tanh of the temps to show approx linear
tfull = np.r_[x, x0]
plt.scatter(tfull, np.tanh(tfull))
plt.axline((0, 0), (delta, delta), linestyle='--', color='red')
plt.xlabel(f"Temp - Mapped to [{-delta}, {delta}]")
plt.ylabel(f"tanh of Temp")

## Simple RNN Case

In [ ]:
# Set up RNN
h0 = tf.reshape(tf.convert_to_tensor(x0, dtype=tf.float32),(-1, 1)) # Initial hidden state
inputs = tf.keras.Input(batch_shape=(None, Nt, 1))
rnn_layer = tf.keras.layers.SimpleRNN(
    1,
    return_sequences=True,
    activation="tanh"
)
hidden = rnn_layer(inputs, initial_state=h0)

outputs = tf.keras.layers.Dense(1)(hidden)
rnn = tf.keras.Model(inputs, outputs)
rnn.compile(loss = "mean_squared_error", optimizer="Adam")
# rnn.summary()

In [ ]:
# Set simple RNN weights 
alpha=np.exp(-k1)

rweights = rnn.get_weights()
rweights[0] = np.array([[(1-alpha)]]) # Input
rweights[1] = np.array([[alpha]]) # Recurrent connection
rweights[2] = np.array([0])    # RNN Cell Bias
rweights[3] = np.array([[1]])  # Dense Output Activation
rweights[4] = np.array([0])  # Dense Output Bias

rnn.set_weights(rweights)

In [ ]:
# Predict
preds = rnn.predict(X, batch_size=X.shape[0], verbose=0)
preds_full = np.concatenate([x0[:, None, None] , preds], axis=1)

### Check accuracy

Approximation error introduced by `tanh(x)` nonlinearity. We will summarize the MSE and visualize the largest errors to show small differences.

In [ ]:
err = np.mean((preds_full - y)**2, axis=1) # average error over Nt per sample
rmse = np.sqrt(np.mean(err))
idx = np.argsort(err.squeeze())[-4:] # Indices of largest errors

In [ ]:
fig, ax = plt.subplots(2, 2, sharex=True, sharey=True)
fig.suptitle(f"Simulated vs RNN Fit - {int(len(idx))} Largest Errors")
for i in range(0, len(idx)):
    ax.flat[i].plot(y[idx[i], :, :], label="Simulated")    
    ax.flat[i].plot(preds_full[idx[i], :, :], label="RNN Fit")

## Time Warp Simple RNN

Modify constant $k$, by scaling factors. Solve temp curves with new k's, then modify simple RNN weights to time warp.

In [ ]:
alpha = 5 # Scaling factor

k2 = k1*alpha
expkt = np.exp(-k2 * t)[None, :] 
y2 = expkt * x0[:, None] + (1 - expkt) * x[:, None] # shape: (N, Nt)
y2 = y2[..., None]
y2 = np.concatenate([x0[:, None, None] , y2], axis=1)

k3 = k1* 1/alpha
expkt = np.exp(-k3 * t)[None, :] 
y3 = expkt * x0[:, None] + (1 - expkt) * x[:, None] # shape: (N, Nt)
y3 = y3[..., None]
y3 = np.concatenate([x0[:, None, None] , y3], axis=1)

In [ ]:
# Plot a few curves with different k

fig, ax = plt.subplots(2, 2)
fig.suptitle(f"Temp Curves with Varying Constants: k=[{k1}, {k2}, {k3}]")
for i in range(0, 4):
    ax.flat[i].plot(y[i, :, :], label = f"k={k1}")
    ax.flat[i].plot(y2[i, :, :], label = f"k={k2}")
    ax.flat[i].plot(y3[i, :, :], label = f"k={k3}")    

handles, labels = ax.flat[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5)
)
plt.tight_layout()